In [ ]:
import pandas as pd

In [ ]:
# load the dataset

df = pd.read_csv("IMDB Dataset.csv")
df.shape

(50000, 2)

### Remove Duplicates from the datsset

In [ ]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [ ]:
df.isnull().sum()  # no null values exits in the dataset

,0
review,0
sentiment,0


### Convert into lowerCase

In [ ]:
df["review"] = df["review"].str.lower()

### remove the URLs

In [ ]:
import re

def remove_URL(text):
  text = re.sub(r"http\S+","",text)
  return text

df["review"] = df["review"].apply(remove_URL)

### Remove punctuations

In [ ]:
def remove_Punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text)
  return text

df["review"] = df["review"].apply(remove_Punctuations)

### Remove html tags

In [ ]:
def remove_html(text):
  text = re.sub(r"<.*?>","",text)
  return text

df["review"] = df["review"].apply(remove_html)

# Do tokenization and Stopword Detection

In [ ]:
!pip install nltk

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download('stopwords')

# punkt-> it is pre trained model for doing tokenization by understanding text structure

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  removed = []
  for token in tokens:
    if token not in stop_words:
      removed.append(token)

  return " ".join(removed)

In [ ]:
df["review"] = df["review"].apply(remove_stopwords)

### now perform stemming

In [ ]:
from nltk.stem import PorterStemmer

# PorterStemmer -> removes suffix of words like es,is,ing

In [ ]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)

  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

sentiments = df["sentiment"]

encoded = le.fit_transform(sentiments)

df["sentiment"] = encoded

In [ ]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product br br film techniqu unass...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


## Perform tfidf vectorization on data

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

y = df["sentiment"]

print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108221 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3128)	0.02247657520221819
  (0, 3699)	0.05303260513845461
  (0, 2820)	0.056335152130759465
  (0, 4829)	0.08683810111930414
  (0, 3186)	0.4779748736819869
  (0, 1548)	0.11363113782724443
  (0, 4988)	0.05847487708069967
  (0, 2188)	0.07920428628671745
  (0, 3722)	0.09124430218990189
  (0, 1596)	0.061875116936704874
  (0, 2067)	0.045550371261069075
  (0, 2800)	0.08207492074377734
  (0, 597)	0.06604637122595439
  (0, 1747)	0.0686964728980499
  (0, 4473)	0.034897814163007954
  (0, 4275)	0.17902125496384533
  (0, 644)	0.07329114041453123
  (0, 3848)	0.03371565199591125
  (0, 4768)	0.25360309572058143
  (0, 3936)	0.044227430566816064
  (0, 4934)	0.0540372278361242
  (0, 1956)	0.03368993773064575
  (0, 4612)	0.07593396995392995
  (0, 3994)	0.14607477308987574
  (0, 2104)	0.05855708556851623
  :	:
  (49581, 2293)	0.10458344189020945
  (49581, 1668)	0.1843322623570

In [ ]:
# train_test_split

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [ ]:
print("Total Values in training dataset: ",X_train.shape," and Total Values in training dataset: ",X_test.shape)

Total Values in training dataset:  (37186, 5000)  and Total Values in training dataset:  (12396, 5000)


In [ ]:
# to convert data to tensors first convert sparse metrices to numpy array

X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# convert to numpy array to tensors


X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32)

In [ ]:
# dataset and DataLoader

from torch.utils.data import TensorDataset,DataLoader

#Datasets
X_train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
X_test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

#DataLoader
X_train_loader = DataLoader(X_train_dataset,batch_size=64,shuffle=True)
X_test_loader = DataLoader(X_test_dataset,batch_size=64,shuffle=True)

In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_state=128,layer=1):
    super().__init__()

    self.input_size = input_size
    self.hidden_state = hidden_state
    self.layer = layer

    self.rnn = nn.RNN(input_size,hidden_state,batch_first=True)

    self.fc = nn.Linear(hidden_state,1)

  def forward(self,x):
    h0 = torch.zeros(self.layer,x.size(0),self.hidden_state)

    #1st Value: hidden state of time steps => [batch_size,seq_len,hidden_size]
    #2st Value: final hiddn state of last hidden state
    out,_ = self.rnn(x,h0)

    out = self.fc(out[:,-1,:])

    return out


In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() #Binary cross entropy loss

import torch.optim as optim

optimizer = optim.Adam(model.parameters())


In [ ]:
epochs=10

for i in range(0,epochs):
  l=0

  model.train()

  for xb,yb in X_train_loader:

    optimizer.zero_grad()

    xb = xb.unsqueeze(1)  #RNN expects 3 dimensional input

    output = model(xb)  #(batch_size,1)

    output = torch.sigmoid(output.squeeze())

    loss = criterion(output,yb)
    loss.backward()  #backward prop
    optimizer.step()  # optimze the loss
    l += loss.item()

  print("Loss ",i+1, "epoch: ",l/len(X_train_loader))

Loss  0 epoch:  0.3468459997841403
Loss  1 epoch:  0.2308147208976377
Loss  2 epoch:  0.21515902579854854
Loss  3 epoch:  0.20607263290185698
Loss  4 epoch:  0.20273551218134842
Loss  5 epoch:  0.19784914357811725
Loss  6 epoch:  0.1950700670685555
Loss  7 epoch:  0.19366960066828326
Loss  8 epoch:  0.1913523845425791
Loss  9 epoch:  0.19014615726839637


In [ ]:
# now evaluate the model

model.eval()

with torch.no_grad():
  total_val =0
  correct =0

  for xb,yb in X_test_loader:

    xb = xb.unsqueeze(1)

    output = model(xb)

    predict = (torch.sigmoid(output.squeeze()) > 0.5).float()
    total_val+=(xb.size(0))

    correct +=(predict==yb).sum().item()

print("Total inputs: ",total_val,"and Total correct: ",correct)

print("Accuracy of the model: ",correct/total_val)

Total inputs:  12396 and Total correct:  10782
Accuracy of the model:  0.8697967086156825
